# Clase 10 · ¿Error o hallazgo?
## Detección de valores atípicos, análisis de distribuciones y transformaciones

**Python para Ciencia de Datos (3.4.258)** · Unidad 4 – Análisis exploratorio de datos
Licenciatura en Inteligencia Artificial y Ciencia de Datos · UADE · Prof. Lic. Ariel Bulacio

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arielbulacio-coder/CinciaDeDatos/blob/main/clase-10-outliers-transformaciones/notebooks/clase10_estudiante.ipynb) &nbsp; *Versión estudiante*

**Al finalizar vas a poder:**
1. **Detectar** valores atípicos con IQR y z-score, y explicar por qué difieren.
2. **Decidir y justificar** qué hacer con cada atípico: corregir, conservar o marcar.
3. **Transformar** variables (log, normalización, discretización) y **crear** nuevas.

> Dataset: 500 hogares **simulados** con la estructura de la Encuesta Permanente de Hogares (EPH – INDEC).
> No son datos reales: fueron diseñados para contener errores de carga y valores extremos genuinos.

## 0 · Preparación del entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Paleta institucional UADE: azul para los datos, ámbar para resaltar
AZUL, AMBAR, TINTA = "#2F6DB5", "#D98600", "#061B2C"
sns.set_theme(style="whitegrid", rc={"axes.titleweight": "bold", "axes.titlelocation": "left"})
pd.options.display.float_format = "{:,.2f}".format

### Generar el dataset
La función crea siempre los mismos 500 hogares (semilla fija → **reproducibilidad**).
En la tarea vas a reemplazar esta celda por los microdatos reales de la EPH.

In [ ]:
SEMILLA = 2026
N_HOGARES = 500
REGIONES = ['GBA', 'Pampeana', 'NOA', 'NEA', 'Cuyo', 'Patagonia']
PESOS_REGION = [0.38, 0.25, 0.11, 0.08, 0.08, 0.1]
FACTOR_REGION = {'GBA': 1.0, 'Pampeana': 0.95, 'NOA': 0.78, 'NEA': 0.74, 'Cuyo': 0.85, 'Patagonia': 1.3}

def generar(n: int = N_HOGARES, semilla: int = SEMILLA) -> pd.DataFrame:
    """Devuelve un DataFrame reproducible de ``n`` hogares simulados."""
    rng = np.random.default_rng(semilla)

    region = rng.choice(REGIONES, size=n, p=PESOS_REGION)
    miembros = np.clip(rng.poisson(2.2, size=n) + 1, 1, 8)
    edad_jefe = np.clip(rng.normal(47, 14, size=n), 18, 90).round().astype(int)
    horas = np.clip(rng.normal(40, 11, size=n), 0, 75).round().astype(int)

    # Ingreso log-normal: asimetría a derecha, típica de variables económicas.
    base = rng.lognormal(mean=np.log(1_150_000), sigma=0.55, size=n)
    factor = np.array([FACTOR_REGION[r] for r in region])
    ingreso = (base * factor * (1 + 0.08 * (miembros - 1))).round(-3)

    df = pd.DataFrame({
        "id_hogar": np.arange(1, n + 1),
        "region": region,
        "miembros": miembros,
        "edad_jefe": edad_jefe,
        "horas_trabajo": horas,
        "ingreso_total": ingreso,
    })

    # --- Atípicos GENUINOS: hogares de muy altos ingresos (existen). ---
    df.loc[[17, 133, 288, 402], "ingreso_total"] = [9_800_000, 12_500_000,
                                                    8_900_000, 15_200_000]
    df.loc[133, "region"] = "Patagonia"   # caso usado en la práctica guiada
    # --- Atípicos por ERROR de carga. ---
    df.loc[56, "ingreso_total"] = 1_350_000 * 100   # dos ceros de más
    df.loc[[91, 377], "edad_jefe"] = 999            # código "no responde"
    df.loc[245, "horas_trabajo"] = 400              # imposible (> 168 h/semana)

    return df


df = generar()
df.head()

### Funciones de la clase
Son las mismas que están en `src/eda_utils.py`, que además tiene tests en `pytest` (Unidad 6).

In [ ]:
def limites_iqr(serie: pd.Series, k: float = 1.5) -> tuple[float, float]:
    """Devuelve los límites de Tukey: (Q1 - k*IQR, Q3 + k*IQR).

    Args:
        serie: valores numéricos (se ignoran los NaN).
        k: multiplicador del rango intercuartílico. 1.5 = atípico, 3 = extremo.
    """
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


def atipicos_iqr(serie: pd.Series, k: float = 1.5) -> pd.Series:
    """Máscara booleana: True donde el valor cae fuera de los límites de Tukey."""
    inferior, superior = limites_iqr(serie, k)
    return (serie < inferior) | (serie > superior)


def atipicos_zscore(serie: pd.Series, umbral: float = 3.0) -> pd.Series:
    """Máscara booleana: True donde |z| > umbral, con z = (x - media) / desvío.

    Advertencia: la media y el desvío se ven afectados por los mismos
    atípicos que se quieren detectar (efecto *enmascaramiento*).
    Supone una distribución aproximadamente normal.
    """
    z = (serie - serie.mean()) / serie.std()
    return z.abs() > umbral


def atipicos_zscore_robusto(serie: pd.Series, umbral: float = 3.5) -> pd.Series:
    """Máscara booleana con z robusto (mediana y MAD), de Iglewicz y Hoaglin.

    z_r = 0.6745 * (x - mediana) / MAD
    """
    mediana = serie.median()
    mad = (serie - mediana).abs().median()
    if mad == 0:
        return pd.Series(False, index=serie.index)
    z_r = 0.6745 * (serie - mediana) / mad
    return z_r.abs() > umbral


def winsorizar(serie: pd.Series, k: float = 1.5) -> pd.Series:
    """Recorta (sin eliminar filas) los valores fuera de los límites de Tukey."""
    inferior, superior = limites_iqr(serie, k)
    return serie.clip(lower=inferior, upper=superior)


def normalizar_minmax(serie: pd.Series) -> pd.Series:
    """Escala al rango [0, 1]: (x - min) / (max - min)."""
    rango = serie.max() - serie.min()
    if rango == 0:
        return pd.Series(0.0, index=serie.index)
    return (serie - serie.min()) / rango


def estandarizar(serie: pd.Series) -> pd.Series:
    """Estandarización z: media 0 y desvío 1."""
    return (serie - serie.mean()) / serie.std()


def resumen_atipicos(df: pd.DataFrame, columnas: list[str]) -> pd.DataFrame:
    """Tabla comparativa: cuántos atípicos detecta cada método en cada columna."""
    filas = []
    for col in columnas:
        s = df[col].dropna()
        filas.append({
            "variable": col,
            "asimetria": round(s.skew(), 2),
            "IQR (k=1.5)": int(atipicos_iqr(s).sum()),
            "z-score (|z|>3)": int(atipicos_zscore(s).sum()),
            "z robusto (>3.5)": int(atipicos_zscore_robusto(s).sum()),
        })
    return pd.DataFrame(filas).set_index("variable")

## 1 · Detectar: antes de calcular, mirá la distribución

In [ ]:
df.describe().T

**Pregunta:** ¿qué te llama la atención del `max` de `edad_jefe`, `horas_trabajo` e `ingreso_total`?

In [ ]:
s = df["ingreso_total"]
print(f"Media:   ${s.mean():>14,.0f}")
print(f"Mediana: ${s.median():>14,.0f}")
print(f"Asimetría (skew): {s.skew():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(s[s < 2e7] / 1e6, bins=40, color=AZUL, ax=axes[0])
axes[0].axvline(s.median() / 1e6, color=TINTA, ls="--", label="mediana")
axes[0].axvline(s.mean() / 1e6, color=AMBAR, ls="--", label="media")
axes[0].set(title="Ingreso (sin el hogar de $135 M)", xlabel="millones de $")
axes[0].legend()
sns.boxplot(x=s / 1e6, color=AZUL, ax=axes[1])
axes[1].set(title="Boxplot del ingreso (con todos los hogares)", xlabel="millones de $")
plt.tight_layout()

**Atípico ≠ error.**
- *Error de carga o medición*: imposible o incoherente (edad 999, 400 h/semana, un cero de más).
- *Extremo genuino*: real pero poco frecuente. Muchas veces es lo más interesante.

### Método 1 · Regla de Tukey (IQR)

In [ ]:
inf, sup = limites_iqr(s)
q1, q3 = s.quantile([.25, .75])
print(f"Q1 = {q1:,.0f}  Q3 = {q3:,.0f}  IQR = {q3 - q1:,.0f}")
print(f"Límites: [{inf:,.0f} ; {sup:,.0f}]")
print("Atípicos por IQR:", atipicos_iqr(s).sum())

### Método 2 · z-score y el efecto enmascaramiento

In [ ]:
print("z-score |z|>3          :", atipicos_zscore(s).sum())
print("z-score sin el hogar 57:", atipicos_zscore(s.drop(56)).sum())
print("z robusto (MAD)        :", atipicos_zscore_robusto(s).sum())

El hogar de \$135 M **infla el desvío estándar** y esconde a los demás: con él, el z-score encuentra uno solo.
Si lo sacamos, aparecen otros. Por eso, con variables asimétricas preferimos **IQR** o **z robusto** (mediana y MAD).

In [ ]:
resumen_atipicos(df, ["ingreso_total", "edad_jefe", "horas_trabajo", "miembros"])

## 2 · Decidir: tres situaciones, tres decisiones

| Situación | Decisión | Ejemplo |
|---|---|---|
| Error evidente | Corregir con la fuente o pasar a `NaN` | `edad_jefe = 999` |
| Extremo genuino | Conservar; usar mediana, log o winsorizar | ingreso \$15,2 M |
| Dudoso | Marcar con una *flag* y analizar con y sin él | ingreso \$135 M |

In [ ]:
limpio = df.copy()

# 1) Errores evidentes → NaN (documentado)
limpio.loc[limpio["edad_jefe"] > 110, "edad_jefe"] = np.nan        # 999 = "no responde"
limpio.loc[limpio["horas_trabajo"] > 168, "horas_trabajo"] = np.nan # una semana tiene 168 h

# 2) Dudoso → flag, y lo excluimos del análisis de ingresos
limpio["flag_ingreso_dudoso"] = limpio["ingreso_total"] > 100_000_000
limpio.loc[limpio["flag_ingreso_dudoso"], "ingreso_total"] = np.nan

limpio.isna().sum()

> 📝 **Registro de decisiones:** cada cambio queda escrito en el notebook. Otra persona tiene que poder reproducirlo.

## 3 · Transformar

### Logaritmo: domar la asimetría

In [ ]:
ing = limpio["ingreso_total"].dropna()
limpio["log_ingreso"] = np.log10(limpio["ingreso_total"])
log_ing = limpio["log_ingreso"].dropna()

print(f"Asimetría original: {ing.skew():.2f} → atípicos IQR: {atipicos_iqr(ing).sum()}")
print(f"Asimetría log10:    {log_ing.skew():.2f} → atípicos IQR: {atipicos_iqr(log_ing).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(ing / 1e6, bins=40, color=AZUL, ax=axes[0]).set(title="Original", xlabel="millones de $")
sns.histplot(log_ing, bins=40, color=AZUL, ax=axes[1]).set(title="log10(ingreso)")
plt.tight_layout()

### Normalizar (min-max) y estandarizar (z)

In [ ]:
edad = limpio["edad_jefe"].dropna()
comparacion = pd.DataFrame({
    "original": edad,
    "minmax": normalizar_minmax(edad),
    "z": estandarizar(edad),
})
comparacion.describe().loc[["mean", "std", "min", "max"]]

In [ ]:
# Lo mismo con scikit-learn (lo usaremos en la Unidad 5)
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

X = limpio[["edad_jefe"]].dropna()
for scaler in (MinMaxScaler(), StandardScaler(), RobustScaler()):
    r = scaler.fit_transform(X)
    print(f"{type(scaler).__name__:15s} min={r.min():6.2f}  max={r.max():6.2f}")

⚠️ El escalado cambia **las unidades, no la forma**: no corrige la asimetría. Para eso, log.

### Discretizar: `pd.cut` vs `pd.qcut`

In [ ]:
limpio["tramo_edad"] = pd.cut(limpio["edad_jefe"], bins=[17, 29, 44, 64, 120],
                              labels=["18–29", "30–44", "45–64", "65+"])
limpio["quintil"] = pd.qcut(limpio["ingreso_total"], q=5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])

print(limpio["tramo_edad"].value_counts().sort_index(), "\n")
print(limpio["quintil"].value_counts().sort_index())

### Crear variables nuevas

In [ ]:
limpio["ingreso_pc"] = limpio["ingreso_total"] / limpio["miembros"]
limpio["flag_atipico"] = atipicos_iqr(limpio["log_ingreso"])

limpio.groupby("quintil", observed=True)["ingreso_pc"].median().map("${:,.0f}".format)

## 4 · Manos al código: retos

**Reto 1 · Detectar.** Encontrá los atípicos de `horas_trabajo` (en `df`, el original) con IQR y con z robusto. ¿Coinciden?

In [ ]:
h = df["horas_trabajo"]
# TODO: mostrá los valores atípicos según IQR y según z robusto

**Reto 2 · Decidir.** Escribí en una celda Markdown qué hiciste con cada atípico de `horas_trabajo` y por qué.

*Tu justificación acá…*

**Reto 3 · Transformar.** Calculá la mediana de `ingreso_pc` por región y ordenala. ¿Cambia el ranking respecto de `ingreso_total`?

In [ ]:
# TODO: agrupá por región, calculá la mediana de ingreso_total e ingreso_pc y compará los rankings

## 5 · Autoevaluación: tests rápidos

In [ ]:
assert limpio["edad_jefe"].max() < 120, "Quedan edades imposibles"
assert limpio["horas_trabajo"].max() <= 168, "Quedan horas imposibles"
assert limpio["flag_ingreso_dudoso"].sum() == 1, "El hogar dudoso debe quedar marcado"
assert {"log_ingreso", "ingreso_pc", "quintil", "tramo_edad"} <= set(limpio.columns)
print("✅ Todo en orden")

## Cierre
1. **Mirar antes de borrar:** un atípico puede ser error o hallazgo.
2. **El método depende de la distribución:** IQR y z robusto resisten; el z-score se enmascara.
3. **Transformar no es maquillar:** log para la asimetría, escalar para comparar, discretizar para interpretar.

**Ticket de salida:** en una frase, ¿por qué el z-score encontró un solo atípico en los ingresos?

**Tarea (aula invertida):** repetí este flujo con los microdatos reales de la EPH (INDEC), variable ITF.
**Lectura:** Bruce, Bruce y Gedeck (2022), cap. 1.